# HG4052 · Week 2 Practical
## From counts to a babbling machine

**No installs, no setup; Week 1's notebook habits are all you need.**

By the end you will have:
- ✅ a bigram heatmap, first for letters and then for phones, with the bright q→u cell found
- ✅ ten pseudo-words babbled by a model that learned English phonotactics from counts alone, and at least five of them that look English
- ✅ perplexity measured on a paragraph the model never saw, and on the same paragraph shuffled, and a reason why one number is higher
- ✅ your first function read aloud: `def` is the f(x) machine from the lecture, written as code

**How this notebook works.** Same as Week 1: click a cell, press **Shift + Enter**, and read the output underneath. Cells marked **✏️ TODO** have one small blank to fill (always one line or less). Everything else is ready to run. Every function in this notebook is provided for you to read and run, never to write.

**Short on time?** Prioritise **Setup → Part 1 → Part 2 → Part 4 → Part 6**. Parts 3 and 5 are quick and worth it, but they can be caught up at home; the Stretch section is take-home by design.


---
## 0 · Setup

Week 1's three commands still work. Where are you standing, and what is here?


In [ ]:
!pwd
!ls

**Imports, and a novel.** NLTK (a Python toolkit for language data, preinstalled on Colab) ships a small Gutenberg collection. We use Jane Austen's *Emma* (1816): about 880,000 characters of English, which is plenty to count. We load it from NLTK's bundled copy, not from gutenberg.org (which blocks cloud machines).


In [ ]:
import math
import random
import csv
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import nltk
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

raw = gutenberg.raw("austen-emma.txt")
print(f"Emma has {len(raw):,} characters. The first 300:\n")
print(raw[:300])

**Get today's data.** The second ingredient is a wordlist: about 7,900 common English words with phonemic transcriptions in IPA (from the CMU Pronouncing Dictionary, with word frequencies from the Brown corpus). One `wget` fetches it into a `data/` folder. The cell checks its own work and tells you whether to use Plan B.


In [ ]:
!mkdir -p data
!wget -q -O data/wordlist_ipa.csv https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week02/wordlist_ipa.csv

import os
if os.path.exists("data/wordlist_ipa.csv") and os.path.getsize("data/wordlist_ipa.csv") > 1000:
    print("✅ Download worked: data/wordlist_ipa.csv is ready. Skip Plan B and carry on.")
else:
    print("❌ Download failed (the file is missing or empty). Use Plan B below.")

In [ ]:
# Plan B (only if the cell above said ❌). The same file, wordlist_ipa.csv, is on NTULearn
# in the Week 2 folder. Download it to your computer, then uncomment the six code lines below
# (delete the leading #) and run: an upload dialog opens, and the file lands in data/.
# (No dialog? Use the folder icon in Colab's left sidebar and drag the file into data/.)
# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, "data/wordlist_ipa.csv")
# print("Now in data/:", os.listdir("data"))

In [ ]:
# Where did it go? -l shows sizes, so an empty file cannot hide:
!ls -l data
!head -5 data/wordlist_ipa.csv

---
## 1 · Read a function aloud

In the lecture, a function was an f(x) machine: something goes in, something comes out. In Python the keyword `def` builds one. The cell below defines two. Before you run it, read `clean_text` aloud, once, in four parts:

| part | in the code | in words |
|---|---|---|
| name | `def clean_text(raw):` | "clean_text is a machine that takes one input, which we call raw" |
| step 1 | `text = raw.lower()` | "make a lowercase copy of the input and call it text" |
| step 2 | `text = remove_punctuation(text)` | "push text through another machine that strips punctuation, keep the result" |
| output | `return text` | "hand back the result to whoever called" |

That is the whole grammar of a function: a name, its inputs in round brackets, an indented block of steps, and `return`. You will never be asked to write one from scratch in this course; from Week 5 on you must recognise them on sight.

`remove_punctuation` is the helper that `clean_text` uses. Read it the same way: it keeps the 26 lowercase letters, turns everything else (punctuation, digits, line breaks) into a space, and squeezes runs of spaces down to one.


In [ ]:
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def remove_punctuation(text):
    kept = "".join(ch if ch in LETTERS else " " for ch in text)   # non-letters become spaces
    return " ".join(kept.split())                                 # squeeze repeated spaces

def clean_text(raw):
    text = raw.lower()
    text = remove_punctuation(text)
    return text

# Nothing happens until the machine is CALLED. Call it on a small string:
print(clean_text("Emma Woodhouse, handsome, clever, and rich..."))

In [ ]:
# ✏️ TODO: call clean_text on this sentence. Shape of the answer:   cleaned = clean_text(sentence)
sentence = "Wreck a nice beach? No: recognize speech!"
cleaned = ...
print("⬆ fill the TODO first, then re-run" if cleaned is ... else cleaned)

In [ ]:
# Now the whole novel goes through the machine: same call, bigger input.
# text.split() then cuts the cleaned string into words at the spaces.
text = clean_text(raw)
words = text.split()

print(f"{len(text):,} characters after cleaning, {len(words):,} word tokens")
print("\nThe first 200 characters:\n")
print(text[:200])

---
## 2 · Count a novel

`Counter` does exactly what its name says: give it a sequence, it counts each distinct item. We feed it every letter of the cleaned novel (skipping the spaces) and ask for the top 12.

Typesetters' folklore says the most frequent letters of English are **E T A O I N**, in that order (the first two columns of keys on a Linotype typesetting machine read ETAOIN SHRDLU). Commit before you run: do you expect Austen to match?


In [ ]:
letter_counts = Counter(ch for ch in text if ch != " ")

print("rank  letter  count")
for rank, (letter, count) in enumerate(letter_counts.most_common(12), start=1):
    print(f"{rank:>4}  {letter:>6}  {count:>6,}")

top6 = "".join(letter for letter, count in letter_counts.most_common(6))
print("\nTop six, in order:", top6.upper(), "   (folklore says ETAOIN)")
assert set(top6) == set("etaoin"), "Same six letters expected; if not, check that text was cleaned"


In [ ]:
# The full picture: all 26 letters, most frequent first
letters_sorted = [letter for letter, count in letter_counts.most_common()]
counts_sorted = [letter_counts[letter] for letter in letters_sorted]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(letters_sorted, counts_sorted, color="#4C72B0")
ax.set_xlabel("letter")
ax.set_ylabel("count in Emma")
ax.set_title("Letter frequencies in Jane Austen's Emma")
plt.show()

**Frequency vs probability.** The bar chart shows frequencies: what you counted. A probability is what you believe about the *next* letter, and the corpus linguist's way to get one is to divide a count by the total: P(letter) = count(letter) / N. That is Goldsmith's distinction from the lecture, and it is one line of code.


In [ ]:
N_letters = sum(letter_counts.values())
print("N (letter tokens in Emma) =", f"{N_letters:,}")

# frequency -> probability, one line:
print("P(t) =", letter_counts["t"] / N_letters)

# ✏️ TODO: the same for e. Shape:   p_e = letter_counts["?"] / N_letters
p_e = ...
if p_e is ...:
    print("P(e) = ⬆ fill the TODO first, then re-run")
else:
    print("P(e) =", round(p_e, 4))
    assert 0.12 < p_e < 0.13, "Expected about 0.126: check the letter and the denominator"
    print("✅ About one letter in eight is an e.")

---
## 3 · Letter bigrams

One letter of memory. A **bigram** is an adjacent pair (previous, next). We count every pair inside every word, and we wrap each word in a boundary symbol `#` so that "word starts with t" and "word ends with e" are pairs too: `("#", "t")` and `("e", "#")`.

The loop below is the engine of this whole practical. Read it before you run it:
- for each word, build the list of symbols: `#`, then its letters, then `#`
- `zip(symbols, symbols[1:])` pairs each symbol with the one after it
- two Counters: `pair_counts` for the pair, `prev_counts` for the first symbol of the pair (the row total you will divide by)


In [ ]:
pair_counts = Counter()     # pair_counts[("t", "h")] = how often h follows t
prev_counts = Counter()     # prev_counts["t"]       = how often t is followed by anything (row total)

for word in words:
    symbols = ["#"] + list(word) + ["#"]
    for prev, nxt in zip(symbols, symbols[1:]):
        pair_counts[(prev, nxt)] += 1
        prev_counts[prev] += 1

N_pairs = sum(pair_counts.values())
print(f"{N_pairs:,} letter pairs counted, {len(pair_counts)} distinct pairs\n")
print("The ten most frequent pairs:")
for (prev, nxt), count in pair_counts.most_common(10):
    print(f"  {prev}{nxt}   {count:,}")

**From counts to conditional probabilities.** The lecture's rule: P(next | previous) = count(previous, next) / count(previous). The numerator is a cell of the table; the denominator is its row total. "Only the denominator moves."

The two functions below are provided: `conditional_matrix` fills a square table of P(next | previous) for a list of symbols, and `show_heatmap` draws it. Rows are the **previous** symbol, columns the **next** symbol, exactly like the 200-pair table on the slide. Bright = likely, dark = rare or impossible. Read the `def` lines and the docstrings, then run.


In [ ]:
def conditional_matrix(pair_counts, prev_counts, symbols):
    """Return a table M where M[i, j] = P(symbols[j] | symbols[i]) = count(pair) / count(previous)."""
    M = np.zeros((len(symbols), len(symbols)))
    for i, prev in enumerate(symbols):
        for j, nxt in enumerate(symbols):
            if prev_counts[prev] > 0:
                M[i, j] = pair_counts[(prev, nxt)] / prev_counts[prev]
    return M


def show_heatmap(M, symbols, title, size=8):
    """Draw M as a heatmap: rows = previous symbol, columns = next symbol, colour = P(next | previous)."""
    fig, ax = plt.subplots(figsize=(size, size * 0.92))
    image = ax.imshow(M, cmap="magma", vmin=0)
    ax.set_xticks(range(len(symbols)))
    ax.set_xticklabels(symbols, fontsize=9)
    ax.set_yticks(range(len(symbols)))
    ax.set_yticklabels(symbols, fontsize=9)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("next symbol")
    ax.set_ylabel("previous symbol")
    ax.set_title(title, pad=14)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03, label="P(next | previous)")
    plt.show()


LETTER_SYMBOLS = ["#"] + list(LETTERS)
M_letters = conditional_matrix(pair_counts, prev_counts, LETTER_SYMBOLS)
show_heatmap(M_letters, LETTER_SYMBOLS, "Letter bigrams in Emma: P(next | previous)")

**Hunt.** Find the brightest cell in the whole table. It is in row `q`. Then look along row `#` (which letters start words?) and down column `#` (which letters end words?).

Two different questions about the same cell, and only the denominator changes:
- **conditional** P(u | q): *given* a q, how often is the next letter u? Numerator: count of the pair qu. Denominator: the q row total, `prev_counts["q"]`.
- **joint** P(q then u): pick any adjacent pair in the novel at random; is it qu? Same numerator. Denominator: all pairs, `N_pairs`.


In [ ]:
print("count(q, u) =", pair_counts[("q", "u")], "   count(q as previous) =", prev_counts["q"], "   N_pairs =", f"{N_pairs:,}")

# ✏️ TODO 1: the CONDITIONAL. Shape:   p_u_given_q = pair_counts[("q", "u")] / prev_counts["q"]
p_u_given_q = ...

# ✏️ TODO 2: the JOINT. Same numerator, the whole world as denominator:   pair_counts[("q", "u")] / N_pairs
p_q_then_u = ...

if p_u_given_q is ...:
    print("P(u | q)    = ⬆ fill TODO 1 first, then re-run")
else:
    print("P(u | q)    =", round(p_u_given_q, 4))
    assert p_u_given_q > 0.9, "P(u | q) should be close to 1: is the denominator the q row total?"

if p_q_then_u is ...:
    print("P(q then u) = ⬆ fill TODO 2 first, then re-run")
else:
    print("P(q then u) =", round(p_q_then_u, 5))
    assert p_q_then_u < 0.01, "P(q then u) should be tiny: is the denominator N_pairs?"
    print("✅ Same cell, two answers: about 1 and about one in a thousand.")

**✏️ One sentence, in your own words (double-click to edit).** Why is P(u | q) so large while P(q then u) is so small?

> Your sentence: **...**


---
## 4 · Phone bigrams (the core)

Letters were the warm-up. The sound system is what we care about, so now the same loop runs over **phones**. The wordlist you downloaded has one line per word: the spelling, its transcription as space-separated IPA phones, and its frequency in the Brown corpus. We split each transcription into a list of phones.

Every word counts **once**, however frequent it is: the question is what English words are allowed to look like, not how often people say them.


In [ ]:
word_phones = []     # one entry per word: (spelling, list of phones, frequency)
with open("data/wordlist_ipa.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        word_phones.append((r["word"], r["ipa"].split(" "), int(r["freq"])))

print(f"Loaded {len(word_phones):,} words. The first five and two from further down:\n")
for word, phones, freq in word_phones[:5] + word_phones[1500:1502]:
    print(f"  {word:<10} {' '.join(phones):<22} freq {freq}")

inventory = sorted(set(p for word, phones, freq in word_phones for p in phones))
print(f"\n{len(inventory)} distinct phones:", " ".join(inventory))

In [ ]:
# For the heatmap axes, the 40 phones in a phonetician's order: monophthongs, diphthongs,
# then consonants by manner, with the boundary # first.
VOWELS = "i ɪ ɛ æ ɑ ɔ ʊ u ʌ ə ɝ eɪ aɪ ɔɪ aʊ oʊ".split()
CONSONANTS = "p b t d k ɡ tʃ dʒ f v θ ð s z ʃ ʒ h m n ŋ l ɹ j w".split()
PHONE_SYMBOLS = ["#"] + VOWELS + CONSONANTS
assert set(PHONE_SYMBOLS[1:]) == set(inventory), "The phone order above must cover exactly the inventory"

phone_pair_counts = Counter()
phone_prev_counts = Counter()

for word, phones, freq in word_phones:               # exactly the Part 3 loop; only the units changed
    symbols = ["#"] + phones + ["#"]
    for prev, nxt in zip(symbols, symbols[1:]):
        phone_pair_counts[(prev, nxt)] += 1
        phone_prev_counts[prev] += 1

print(f"{sum(phone_pair_counts.values()):,} phone pairs counted\n")
M_phones = conditional_matrix(phone_pair_counts, phone_prev_counts, PHONE_SYMBOLS)
show_heatmap(M_phones, PHONE_SYMBOLS, "Phone bigrams, 7,900 English words: P(next | previous)", size=12)

**Read the table like a phonologist.** Four cells to look up, then a hunt.

- **P(ɹ | t)**, the lecture's example: row `t`, column `ɹ`. How often is a t followed by ɹ?
- **P(l | t)**: row `t`, column `l`. English words do not *begin* with /tl/, but this table counts pairs anywhere inside a word, and "atlas", "outline", "exactly" all contain /tl/. Expect small, not zero. (Zero would need a table of word-initial pairs only: a different question, a different count.)
- **P(ŋ | #)**: row `#`, column `ŋ`. Does any English word in the list start with /ŋ/?
- **P(# | ŋ)**: row `ŋ`, column `#`. Given an /ŋ/, how often is the word over?

Same shape as Part 3: `phone_pair_counts[(previous, next)] / phone_prev_counts[previous]`. Copy-paste the IPA symbols from this cell if they are hard to type.


In [ ]:
# ✏️ TODO: four conditional probabilities. Shape:   phone_pair_counts[("t", "ɹ")] / phone_prev_counts["t"]
p_r_given_t = ...
p_l_given_t = ...
p_ng_given_start = ...      # P(ŋ | #)
p_end_given_ng = ...        # P(# | ŋ)

checks = [
    ("P(ɹ | t)", p_r_given_t, 0.05, 0.15, "row t, column ɹ: expect a bit under 0.1"),
    ("P(l | t)", p_l_given_t, 0.0, 0.05, "row t, column l: expect small but not zero"),
    ("P(ŋ | #)", p_ng_given_start, 0.0, 0.0, "row #, column ŋ: expect exactly 0"),
    ("P(# | ŋ)", p_end_given_ng, 0.6, 0.9, "row ŋ, column #: expect about 0.77"),
]
for name, value, low, high, hint in checks:
    if value is ...:
        print(f"{name:<10} = ⬆ fill the TODO first, then re-run")
    else:
        print(f"{name:<10} = {value:.4f}")
        assert low <= value <= high, f"{name} looks wrong. Hint: {hint}"


**Dark-zone hunt.** Dark cells are pairs that (almost) never occur. Start with the large dark square at the top left (vowel rows, vowel columns). Then look along row `#` for phones that never start a word, down column `#` for phones that never end one, and at row `h`. Then answer below.

**✏️ Which phonotactic constraints of English can you read straight off the heatmap?** List three, each as "X never/rarely follows Y" or "no word begins/ends with X". (Double-click to edit.)

> 1. **...**
> 2. **...**
> 3. **...**

One thing the table cannot tell you: whether a dark cell is a rule of English or just an accident of a 7,900-word list. That question returns in Week 6.


---
## 5 · Babble: sample pseudo-words from the model

A table of conditional probabilities is also a **generator**. Start at `#`. Look up the `#` row of the table: it is a probability distribution over what comes next. Roll dice weighted by that row, write down the phone you got, move to its row, roll again, and stop when the dice say `#`. That is the chain rule from the lecture, run forwards, with a one-phone memory.

`random.choices(symbols, weights=probs)` is the weighted dice. The `babble` function below is provided; read the docstring and the loop, then run. `random.seed(9)` fixes the random numbers so that everyone in the room, and you tomorrow, get the same ten words.


In [ ]:
def babble(pair_counts, prev_counts, symbols):
    """Sample one pseudo-word from a bigram model: start at '#', roll the dice of the current row, stop at '#'."""
    out = []
    prev = "#"
    while len(out) < 30:                                                         # safety stop, never reached in practice
        probs = [pair_counts[(prev, s)] / prev_counts[prev] for s in symbols]    # one row of the heatmap
        nxt = random.choices(symbols, weights=probs)[0]                          # weighted dice
        if nxt == "#":
            break
        out.append(nxt)
        prev = nxt
    return out


random.seed(9)
print("Ten pseudo-words from the phone bigram model:\n")
for i in range(10):
    phones = babble(phone_pair_counts, phone_prev_counts, PHONE_SYMBOLS)
    compact = "/" + "".join(phones) + "/"
    print(f"  {i + 1:>2}.  {compact:<16} ({' '.join(phones)})")

# The same machine, fed the LETTER table instead. Nothing in babble() knows what a phone is:
random.seed(9)
print("\nFive strings from the letter bigram model:", ", ".join("".join(babble(pair_counts, prev_counts, LETTER_SYMBOLS)) for _ in range(5)))

**✏️ Which of the ten look like possible English words, and why?** Pick your five best and your one worst, and name the phonotactic reason for the worst (a cluster English does not allow? a word with no vowel? too long?). Double-click to edit.

> Best five: **...**
>
> Worst, and why: **...**

Notice what the model never saw: no rules, no syllable structure, no phoneme inventory, no "English words must contain a vowel". It saw counts of adjacent pairs. Everything English-like in its output was absorbed from those counts. Everything un-English in its output comes from its one-phone memory: each pair is fine, but the model cannot see two phones back. (The stretch section gives it two phones of memory.)


---
## 6 · Perplexity on held-out text

**The held-out rule.** We always evaluate the model on text it never saw. The text it counted is its **training set** (its textbook); the text it is tested on is the **held-out test set** (the exam). Acing your own textbook proves memory, not knowledge. From here to Week 13, every number we trust is measured on held-out data.

So: train a letter bigram model on the first 90% of *Emma*, then score it on one real paragraph from the last 10%, and on the same paragraph with its letters shuffled (same letters, same counts, no structure).

**Perplexity** is the score: 2 to the power of the average surprisal per symbol, the "average branching factor" from the lecture. The four functions below are provided: read each docstring, then run. (One detail: `bigram_prob` adds 0.5 to every count so that a pair the model never saw gets a small probability instead of zero; log₂ of zero is minus infinity, which would wreck the sum. This is *smoothing*; Week 6 does it properly.)


In [ ]:
def train_bigram(text):
    """Count letter bigrams in text, each word wrapped in '#'. Returns (pair_counts, prev_counts): the model."""
    pair_counts = Counter()
    prev_counts = Counter()
    for word in text.split():
        symbols = ["#"] + list(word) + ["#"]
        for prev, nxt in zip(symbols, symbols[1:]):
            pair_counts[(prev, nxt)] += 1
            prev_counts[prev] += 1
    return pair_counts, prev_counts


def bigram_prob(prev, nxt, model, k=0.5):
    """P(nxt | prev) under the model, with add-k smoothing: (count + k) / (row total + k * number of symbols)."""
    pair_counts, prev_counts = model
    V = len(LETTER_SYMBOLS)
    return (pair_counts[(prev, nxt)] + k) / (prev_counts[prev] + k * V)


def log2_prob(text, model):
    """Sum of log2 P(nxt | prev) over every bigram in text (products become sums). Also returns N, the number of predictions made."""
    total = 0.0
    N = 0
    for word in text.split():
        symbols = ["#"] + list(word) + ["#"]
        for prev, nxt in zip(symbols, symbols[1:]):
            total += math.log2(bigram_prob(prev, nxt, model))
            N += 1
    return total, N


def perplexity(text, model):
    """2 ** (average surprisal per symbol) = 2 ** (-(1/N) * sum of log2 P)."""
    total, N = log2_prob(text, model)
    return 2 ** (-(1 / N) * total)


print("Four machines built. Nothing runs until they are called.")

In [ ]:
cut = int(0.9 * len(raw))                       # character position 90% of the way through the novel
train_text = clean_text(raw[:cut])              # the textbook
model = train_bigram(train_text)

# The exam: one real paragraph from the last 10% (the first one between 600 and 700 characters long)
test_paragraphs = [p.strip() for p in raw[cut:].split("\n\n")]
held_out_raw = [p for p in test_paragraphs if 600 <= len(p) <= 700][0]
held_out = clean_text(held_out_raw)

print(f"Training text: {len(train_text):,} characters. Held-out paragraph ({len(held_out)} characters after cleaning):\n")
print(held_out)

**✏️ Commit before you run.** Two perplexities are about to be printed: the real paragraph, and the same paragraph with its letters shuffled. Which will be higher, and roughly by how much? Write your bet here (double-click), then run.

> My bet: **...**


In [ ]:
random.seed(4052)
chars = list(held_out)
random.shuffle(chars)                            # same letters, same counts, random order
shuffled = "".join(chars)

print("Shuffled paragraph starts:", repr(shuffled[:80]), "\n")
ppl_real = perplexity(held_out, model)
ppl_shuffled = perplexity(shuffled, model)
print(f"perplexity on the real held-out paragraph : {ppl_real:6.2f}")
print(f"perplexity on the shuffled paragraph      : {ppl_shuffled:6.2f}")
print(f"perplexity of a model that guesses uniformly over {len(LETTER_SYMBOLS)} symbols : {len(LETTER_SYMBOLS):.2f}  (for reference)")

**What the numbers say.** On the real paragraph the model is about as confused as someone choosing among 11 equally likely symbols: one letter of memory already removes most of the branching. On the shuffled paragraph its perplexity is far higher, and higher even than the 27 of a model that guesses uniformly. The shuffled text has the same letters, so a unigram model would not notice the difference; the bigram model does, because it bet on English *structure* (th, he, e#), and the shuffle destroyed the structure while keeping the letters. Confidently wrong bets cost more than no bets at all.

The model never saw this paragraph. That is the point: we measured what it learned about English, not what it memorised about *Emma*. In Week 6 the same number grades word-level language models inside a recognizer.

**Entropy of the letters themselves.** Average surprisal of the unigram letter distribution from Part 2: H = −Σ P(letter) log₂ P(letter). The lecture quoted 4.70 bits if all 26 letters were equally likely and about 4.1 bits for real English letter frequencies. Check Austen:


In [ ]:
letter_probs = [count / N_letters for count in letter_counts.values()]
H_letters = -sum(p * math.log2(p) for p in letter_probs)

print(f"entropy of Emma's letters (unigram): {H_letters:.2f} bits per letter")
print(f"maximum possible, 26 equally likely letters: {math.log2(26):.2f} bits")
print(f"lecture's figure for English letter frequencies: about 4.1 bits")
print(f"\nas a branching factor: 2 ** {H_letters:.2f} = {2 ** H_letters:.1f} equally likely letters (unigram),")
print(f"versus the bigram model's {ppl_real:.1f} on the held-out paragraph: one letter of memory buys a lot.")

---
## 7 · Stretch (or take-home)

### (a) Two phones of memory: the trigram babbler

The bigram babbler cannot see two phones back, which is why it produced clusters no English word has. A **trigram** model conditions on the previous *two* symbols: P(next | previous two). The functions below are provided: `train_trigram` counts triples (each word now starts with two `#`), `babble_trigram` samples from them. Your only job is to call it.


In [ ]:
def train_trigram(sequences):
    """Count (prev2, prev1, next) triples over a list of symbol sequences. Returns {(prev2, prev1): Counter of next}."""
    followers = {}
    for seq in sequences:
        symbols = ["#", "#"] + list(seq) + ["#"]
        for a, b, c in zip(symbols, symbols[1:], symbols[2:]):
            followers.setdefault((a, b), Counter())[c] += 1
    return followers


def babble_trigram(followers):
    """Sample one pseudo-word from a trigram model: the dice depend on the last TWO symbols."""
    out = []
    prev2, prev1 = "#", "#"
    while len(out) < 30:
        options = followers[(prev2, prev1)]
        nxt = random.choices(list(options.keys()), weights=list(options.values()))[0]
        if nxt == "#":
            break
        out.append(nxt)
        prev2, prev1 = prev1, nxt
    return out


trigram_model = train_trigram([phones for word, phones, freq in word_phones])

random.seed(9)
for i in range(10):
    # ✏️ TODO: call the trigram babbler. Shape:   phones = babble_trigram(trigram_model)
    phones = ...
    print(f"  {i + 1:>2}. ", "⬆ fill the TODO first, then re-run" if phones is ... else f"/{''.join(phones)}/")

Compare with the ten bigram words in Part 5. Better? The price is more parameters to count (every pair of previous phones needs its own row), and rows that were never seen in 7,900 words. Week 6 returns to that trade-off.

### (b) Letters vs phones: which writing system is the corpus hiding?

Part 2 gave the entropy of English *letters* (from the novel). The wordlist lets us compute the entropy of English *phones*: count every phone of every word, weighted by the word's frequency so that common words count more (an approximation of running speech). Before you run: which do you expect to be higher, and why? The symbol sets differ in size (26 letters vs 40 phones), so the cell also prints each entropy against its own maximum.


In [ ]:
phone_counts = Counter()
for word, phones, freq in word_phones:
    for p in phones:
        phone_counts[p] += freq                 # weighted by word frequency

N_phones = sum(phone_counts.values())
H_phones = -sum((c / N_phones) * math.log2(c / N_phones) for c in phone_counts.values())

print("Most frequent phones (frequency-weighted):", ", ".join(f"{p} {c / N_phones:.3f}" for p, c in phone_counts.most_common(6)))
print(f"\nentropy of English letters (Emma):     {H_letters:.2f} bits, out of a possible {math.log2(26):.2f}  ({H_letters / math.log2(26):.0%} of maximum)")
print(f"entropy of English phones (wordlist):  {H_phones:.2f} bits, out of a possible {math.log2(40):.2f}  ({H_phones / math.log2(40):.0%} of maximum)")

# For the discussion below: the phone-table cousin of the letter table's q->u cell
print(f"\nP(w | k) in the phone table = {phone_pair_counts[('k', 'w')] / phone_prev_counts['k']:.3f}   (P(u | q) in the letter table was 1.0)")

**✏️ Discuss (or write a few lines here).** The novel is a corpus of *spelling*; the wordlist is a corpus of *sound*. Look back at your two heatmaps. Which certainties of the letter table are facts about English sounds, and which are facts about English spelling? (Hint: P(u | q) was 1.0 in letters; now look up P(w | k) in the phone table, since "qu" is pronounced /kw/. Then think about "th" and "sh".) Which of the two tables would a speech recognizer need, and why?

> **...**


---
## ✅ Done looks like

- a letter bigram heatmap with the bright q→u cell found, and a phone bigram heatmap with its dark zones named
- ten pseudo-words from the phone model, five of them English-looking, and a phonotactic reason for the worst one
- perplexity higher on the shuffled paragraph than on the real one, and you can say why
- `clean_text` read aloud: name, input, steps, output

**What you built.** A language model. A small one (letters or phones, one symbol of memory), but the real thing: it assigns a probability to any sequence, it can generate sequences, and it can be scored on held-out data by perplexity. In Week 6 this exact object, built over words instead of phones, is the language model inside a speech recognizer: the prior in Bayes' rule, the box that makes "recognize speech" beat "wreck a nice beach". Keep this notebook.
